[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_05/SFM_ch5_gev_block_maxima/SFM_ch5_gev_block_maxima.ipynb)

# SFM_ch5_gev_block_maxima

Block maxima and the generalised extreme value (GEV) distribution: densities and distribution functions of the Gumbel, Frechet and Weibull laws (as in SFEevt1); the Fisher-Tippett-Gnedenko theorem in a simulation (maxima of Exponential, Pareto and Uniform samples); maximum likelihood GEV fits to the monthly maxima of daily losses of the S&P 500, DAX, BET and Bitcoin with standard errors; PP and QQ plots (as in SFEtailGEV_pp and SFEtailGEV_qq); return levels for 1, 10 and 50 years with bootstrap intervals (as in block_max).

*Statistics of Financial Markets (SFM), Chapter 5: heavy tails and extreme value theory. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
from scipy import optimize

START = {'sp500': '1990-01-01', 'dax': '1990-01-01', 'bet': '1997-01-01', 'btc': None, 'snp': '2010-01-01', 'brd': '2010-01-01', 'tlv': '2010-01-01', 'tgn': '2010-01-01'}

ASSETS = ['sp500', 'dax', 'bet', 'btc']

STOCKS = ['snp', 'brd', 'tlv', 'tgn']

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'snp': '#2E7D32', 'brd': '#8E44AD', 'tlv': '#E67E22', 'tgn': '#DC3545'}

MODEL_COL = {'Data': '#B5853F', 'Normal': '#CD0000', 'EVT': '#1A3A6E', 'Historical': '#2E7D32', 'Hill': '#8E44AD'}

SEED = 2026

HILL_FRAC = 0.025

POT_Q = 0.9

SPLIT = '2019-12-31'

BLOCK = 20

def returns(k):
    """Daily log returns in % on the series' own calendar (S&P 500 and DAX since 1990, BET since its launch in
    1997, Bitcoin since 2014, BVB stocks since 2010); known data errors removed."""
    r = log_returns(k, START.get(k))
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def losses(k):
    """Daily losses L_t = -r_t in % (a positive number is a loss)."""
    return -returns(k)


def num_hessian(f, p, h=1e-4):
    """Numerical Hessian of f at p (central differences)."""
    p = np.asarray(p, float)
    k = len(p)
    H = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            e_i, e_j = np.eye(k)[i] * h, np.eye(k)[j] * h
            H[i, j] = (f(p + e_i + e_j) - f(p + e_i - e_j) - f(p - e_i + e_j) + f(p - e_i - e_j)) / (4 * h * h)
    return H


def gev_cdf(x, xi, mu, sigma):
    """GEV distribution function H(x) = exp{-[1 + xi (x - mu)/sigma]^(-1/xi)} (xi = 0: Gumbel)."""
    return stats.genextreme.cdf(x, -xi, mu, sigma)          # scipy's shape c = -xi


def gev_ppf(p, xi, mu, sigma):
    return stats.genextreme.ppf(p, -xi, mu, sigma)


def gev_pdf(x, xi, mu, sigma):
    return stats.genextreme.pdf(x, -xi, mu, sigma)


def fit_gev(m):
    """Maximum likelihood GEV fit (xi, mu, sigma) of block maxima m, standard errors from the numerical Hessian."""
    m = np.asarray(m, float)
    c, mu, sig = stats.genextreme.fit(m, -0.1, loc=np.mean(m), scale=np.std(m))
    nll = lambda p: -np.sum(stats.genextreme.logpdf(m, -p[0], p[1], p[2])) if p[2] > 0 else np.inf
    res = optimize.minimize(nll, [-c, mu, sig], method='Nelder-Mead', options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 5000})
    xi, mu, sig = res.x
    cov = np.linalg.inv(num_hessian(nll, res.x))
    se = np.sqrt(np.diag(cov))
    return {'xi': xi, 'mu': mu, 'sigma': sig, 'se_xi': se[0], 'se_mu': se[1], 'se_sigma': se[2], 'nll': res.fun, 'n': len(m)}


def return_level(g, months):
    """Level exceeded by the monthly maximum on average once every `months` months: z = H^(-1)(1 - 1/months)."""
    return float(gev_ppf(1 - 1 / months, g['xi'], g['mu'], g['sigma']))


def fig_gev_types(save=True):
    """Densities and distribution functions of the three extreme value laws (as in SFEevt1): Gumbel (xi = 0),
    Frechet (xi = 0.5) and Weibull (xi = -0.5), standard location and scale."""
    x = np.linspace(-4, 7, 600)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for xi, lab, c in [(0.0, 'Gumbel (xi = 0)', '#1A3A6E'), (0.5, 'Frechet (xi = 0.5)', '#CD0000'),
                       (-0.5, 'Weibull (xi = -0.5)', '#2E7D32')]:
        axes[0].plot(x, gev_pdf(x, xi, 0, 1), color=c, lw=1.6, label=lab)
        axes[1].plot(x, gev_cdf(x, xi, 0, 1), color=c, lw=1.6, label='_')
    axes[0].set_title('Density h(x)', loc='left', fontsize=12)
    axes[1].set_title('Distribution function H(x)', loc='left', fontsize=12)
    for ax in axes:
        ax.set_xlabel('x')
    st.fig_legend_bottom(fig, ncol=3, y=0.0)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch5_gev_types')


def fig_maxima_sim(n=250, reps=5000, save=True):
    """Fisher-Tippett-Gnedenko in a simulation: maxima of n = 250 draws, normalised, against the limit law.
    Exponential(1): M - ln n -> Gumbel; Pareto(3): M / n^(1/3) -> Frechet; Uniform(0, 1): n (M - 1) -> Weibull."""
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
    cases = [('Exponential(1): M - ln n', lambda: rng.exponential(1, (reps, n)).max(1) - np.log(n), 0.0, '#1A3A6E', (-3, 7)),
             ('Pareto(3): M / n^(1/3)', lambda: (rng.pareto(3, (reps, n)) + 1).max(1) / n ** (1 / 3), 1 / 3, '#CD0000', (0, 4)),
             ('Uniform(0, 1): n (M - 1)', lambda: n * (rng.uniform(0, 1, (reps, n)).max(1) - 1), -1.0, '#2E7D32', (-5, 0.2))]
    out = {}
    for ax, (lab, draw, xi, c, lim) in zip(axes, cases):
        m = draw()
        x = np.linspace(*lim, 400)
        bins = np.linspace(*lim, 50)
        ax.hist(m, bins=bins, density=True, color=c, alpha=0.35, label='_')
        if xi == 0:
            f = gev_pdf(x, 0, 0, 1)
        elif xi > 0:
            f = gev_pdf(x, xi, 1, xi)                     # Frechet: P(M/n^(1/3) <= x) -> exp(-x^-3) = GEV(1/3, 1, 1/3)
        else:
            f = gev_pdf(x, -1, -1, 1)                    # Weibull: P(n(M-1) <= x) -> exp(x) for x <= 0 = GEV(-1, -1, 1)
        ax.plot(x, f, color='black', lw=1.4, label='Limit law (GEV)')
        ax.set_title(lab, loc='left', fontsize=11)
        out[lab] = float(np.mean(m))
    axes[0].bar([0], [0], color='#1A3A6E', alpha=0.35, label='Simulated maxima (5,000 samples)')
    st.fig_legend_bottom(fig, ncol=2, y=0.0)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch5_maxima_sim')
    return out


def monthly_maxima(k):
    """Largest daily loss in each calendar month (blocks of about 21 trading days; 30 days for Bitcoin)."""
    L = losses(k)
    g = L.groupby([L.index.year, L.index.month])
    m = g.max()[g.count() >= 15]
    return m


def gev_table(names=ASSETS, B=200, seed=SEED):
    """GEV fit to the monthly maxima of daily losses; 10-year and 50-year return levels with bootstrap intervals."""
    rng = np.random.default_rng(seed)
    out = {}
    for k in names:
        m = monthly_maxima(k).values
        g = fit_gev(m)
        rl10, rl50 = return_level(g, 120), return_level(g, 600)
        bs = []
        for _ in range(B):
            s = rng.choice(m, len(m))
            try:
                c, mu, sig = stats.genextreme.fit(s, -g['xi'], loc=g['mu'], scale=g['sigma'])
                bs.append([-c, stats.genextreme.ppf(1 - 1 / 120, c, mu, sig)])
            except Exception:
                pass
        bs = np.array(bs)
        out[k] = dict(g, rl10=rl10, rl50=rl50, rl1=return_level(g, 12), rl10_lo=float(np.percentile(bs[:, 1], 2.5)),
                      rl10_hi=float(np.percentile(bs[:, 1], 97.5)), xi_lo=g['xi'] - 1.96 * g['se_xi'],
                      xi_hi=g['xi'] + 1.96 * g['se_xi'], max=float(m.max()), months=len(m),
                      years=len(m) / 12, n_above_rl10=int((m > rl10).sum()))
    return out


def fig_gev_ppqq(k='sp500', save=True):
    """PP and QQ plots of the GEV fitted to the monthly maxima of daily losses (as in SFEtailGEV_pp and _qq)."""
    m = np.sort(monthly_maxima(k).values)
    g = fit_gev(m)
    p = (np.arange(1, len(m) + 1) - 0.5) / len(m)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
    axes[0].plot(p, gev_cdf(m, g['xi'], g['mu'], g['sigma']), 'o', ms=3, color=COLORS[k], label='Monthly maxima')
    axes[0].plot([0, 1], [0, 1], color='black', lw=0.8, ls='--', label='45-degree line')
    axes[0].set_xlabel('Empirical probability')
    axes[0].set_ylabel('GEV probability')
    axes[0].set_title('PP plot', loc='left', fontsize=12)
    q = gev_ppf(p, g['xi'], g['mu'], g['sigma'])
    axes[1].plot(q, m, 'o', ms=3, color=COLORS[k], label='_')
    lim = [min(q.min(), m.min()), max(q.max(), m.max())]
    axes[1].plot(lim, lim, color='black', lw=0.8, ls='--', label='_')
    axes[1].set_xlabel('GEV quantile (%)')
    axes[1].set_ylabel('Monthly maximum of daily losses (%)')
    axes[1].set_title('QQ plot', loc='left', fontsize=12)
    st.fig_legend_bottom(fig, ncol=2, y=0.0)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig(f'sfm_ch5_gev_ppqq_{k}' if k != 'sp500' else 'sfm_ch5_gev_ppqq')
    return g


def fig_return_levels(gt, names=ASSETS, save=True):
    """Return level plot: the monthly maximum of daily losses exceeded on average once in T years (GEV fit)
    against the empirical return levels of the observed monthly maxima."""
    fig, ax = plt.subplots(figsize=(10, 4.2))
    T = np.logspace(np.log10(1 / 12 * 1.05), np.log10(100), 200)
    for k in names:
        g = gt[k]
        z = gev_ppf(1 - 1 / (12 * T), g['xi'], g['mu'], g['sigma'])
        ax.semilogx(T, z, color=COLORS[k], lw=1.6, label=f'{LABELS[k]} (GEV)')
        m = np.sort(monthly_maxima(k).values)
        pe = (np.arange(1, len(m) + 1)) / (len(m) + 1)
        ax.semilogx(1 / (12 * (1 - pe)), m, 'o', ms=2.5, color=COLORS[k], alpha=0.6, label='_')
    ax.axvline(10, color='black', lw=0.8, ls=':', label='10-year return period')
    ax.set_xlabel('Return period T (years, log scale)')
    ax.set_ylabel('Return level: daily loss (%)')
    st.legend_outside_bottom(ax, ncol=5, y=-0.16)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch5_return_levels')

## Run

In [ ]:
fig_gev_types()
print(fig_maxima_sim())
gt = gev_table()
print(pd.DataFrame(gt).T[['months', 'xi', 'se_xi', 'mu', 'sigma', 'rl1', 'rl10', 'rl10_lo', 'rl10_hi', 'rl50', 'max']].round(2))
fig_gev_ppqq('sp500')
fig_return_levels(gt)

![sfm_ch5_gev_types](./sfm_ch5_gev_types.png)

Output: `sfm_ch5_gev_types.pdf`

![sfm_ch5_maxima_sim](./sfm_ch5_maxima_sim.png)

Output: `sfm_ch5_maxima_sim.pdf`

![sfm_ch5_gev_ppqq](./sfm_ch5_gev_ppqq.png)

Output: `sfm_ch5_gev_ppqq.pdf`

![sfm_ch5_return_levels](./sfm_ch5_return_levels.png)

Output: `sfm_ch5_return_levels.pdf`